
# Thí nghiệm 01 — Huấn luyện mô hình dự báo hệ Lorenz

## Mục tiêu

Notebook này ghi lại quy trình huấn luyện ba biến thể mô hình dự báo chuỗi thời gian Lorenz, sử dụng các bộ trích xuất đặc trưng dựa trên PatchTST, Autoformer và Informer kết hợp với toán tử Koopman ổn định.

Quy trình thực nghiệm bao gồm:

1. Thiết lập môi trường và nạp bộ dữ liệu Lorenz đã chuẩn bị.
2. Xác định hàm mất mát kết hợp sai số dự báo và chính quy hóa ổn định.
3. Huấn luyện ba mô hình với cùng cấu hình cơ bản.
4. Lưu trọng số mô hình và lịch sử huấn luyện.
5. Phân tích các kết quả dự báo và đặc tính ổn định đã thu được.

## Phạm vi

Notebook lưu lại mã nguồn huấn luyện và các kết quả từ lần thực nghiệm đã hoàn thành.

Các bảng số liệu và hình minh họa sẽ được tập hợp riêng trong những notebook đánh giá và trực quan hóa.



## 1. Thiết lập môi trường và nạp dữ liệu

### 1.1. Môi trường thực nghiệm

Notebook lựa chọn GPU thông qua CUDA nếu thiết bị hỗ trợ, ngược lại sử dụng CPU.

Output đã lưu ghi nhận quá trình thực nghiệm sử dụng GPU NVIDIA Quadro T1000.

### 1.2. Nạp kiến trúc mô hình

Các định nghĩa mô hình được nạp từ `02_lorenz_model.ipynb`.

Cách tổ chức này giúp tách phần xây dựng kiến trúc khỏi quá trình huấn luyện.

### 1.3. Nạp dữ liệu

Bộ dữ liệu được đọc từ:

`results/lorenz_dataset.pt`

Các thành phần được sử dụng gồm:

| Thành phần | Kích thước hoặc giá trị |
|---|---|
| Đầu vào `X` | (1845, 150, 3) |
| Nhãn `Y` | (1845, 15) |
| Độ dài đầu vào | 150 |
| Chân trời dự báo | 5 |

Dữ liệu đã được tạo ở notebook chuẩn bị dữ liệu, vì vậy không cần thực hiện lại quá trình mô phỏng Lorenz.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

# 1. Xác định thiết bị tính toán
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# 2. Xác định đường dẫn thí nghiệm
cwd = Path.cwd()

if (cwd / "experiments" / "01_lorenz").is_dir():
    experiment_dir = cwd / "experiments" / "01_lorenz"
elif cwd.name == "01_lorenz":
    experiment_dir = cwd
else:
    raise FileNotFoundError("Không tìm thấy thư mục 01_lorenz")

# 3. Nạp các định nghĩa mô hình
model_path = experiment_dir / "02_lorenz_model.ipynb"

get_ipython().run_line_magic(
    "run",
    f'"{model_path.as_posix()}"'
)

# 4. Nạp bộ dữ liệu Lorenz đã chuẩn bị
dataset = torch.load(
    experiment_dir / "results" / "lorenz_dataset.pt",
    map_location="cpu"
)

x_in = dataset["X"].to(device)
x_out = dataset["Y"].to(device)

patch_length = dataset["patch_length"]
horizon = dataset["horizon"]

# 5. Hiển thị thông tin bộ dữ liệu
print("\n=== Dataset ===")
print("X:", x_in.shape)
print("Y:", x_out.shape)
print("Patch Length:", patch_length)
print("Horizon:", horizon)
print("Results Directory:", experiment_dir / "results")

Device: cuda
GPU: Quadro T1000
Input z shape: torch.Size([4, 16])
Output z_next shape: torch.Size([4, 16])
Koopman matrix K shape: torch.Size([16, 16])
Singular values S shape: torch.Size([16])

=== Stability Check ===
Max singular value: 0.810895562171936
Spectral radius: 0.4780125617980957
Stable: True

=== Latent Norm Check ===
Before Koopman: tensor([4.4333, 5.0287, 3.7499, 3.9687])
After Koopman: tensor([2.1793, 2.4723, 1.5100, 1.9987], grad_fn=<LinalgVectorNormBackward0>)
Contraction satisfied: True
Input shape: torch.Size([2, 150, 3])
Latent shape: torch.Size([2, 16])

=== Dataset ===
X: torch.Size([1845, 150, 3])
Y: torch.Size([1845, 15])
Patch Length: 150
Horizon: 5
Results Directory: c:\Users\Win 11 New\Desktop\pa gk\experiments\01_lorenz\results



## 2. Huấn luyện mô hình PatchTST kết hợp Koopman

### 2.1. Mục tiêu

Thí nghiệm thực hiện huấn luyện mô hình `PatchTSTWrapper` trên bộ dữ liệu Lorenz đã chuẩn bị.

Mô hình sử dụng backbone dựa trên PatchTST để trích xuất biểu diễn tiềm ẩn, sau đó áp dụng toán tử Koopman ổn định và bộ giải mã tuyến tính để dự báo các giá trị tương lai.

### 2.2. Cấu hình huấn luyện

| Tham số | Giá trị |
|---|---|
| Mô hình | `PatchTSTWrapper` |
| Số mẫu huấn luyện | 1.845 |
| Số epoch | 3.000 |
| Batch size | 1.845 (full-batch) |
| Bộ tối ưu | Adam |
| Learning rate | 0.001 |
| Hệ số chính quy hóa | 0.1 |
| Random seed | 42 |

### 2.3. Hàm mất mát

Hàm mất mát được tính trực tiếp trong vòng lặp huấn luyện, gồm hai thành phần:

- **Sai số dự báo (MSE):** đo mức chênh lệch giữa giá trị dự báo và nhãn.
- **Chính quy hóa ổn định:** phạt trường hợp bình phương chuẩn của biểu diễn tiềm ẩn tăng sau phép biến đổi Koopman.

Hàm mất mát tổng hợp:

$$
\mathcal{L}
=
\mathcal{L}_{\mathrm{MSE}}
+
0.1\mathcal{L}_{\mathrm{stability}}
$$

Trong đó:

$$
\mathcal{L}_{\mathrm{stability}}
=
\frac{1}{B}\sum_{i=1}^{B}
\max\left(
0,
\|\mathbf{z}'_i\|_2^2-\|\mathbf{z}_i\|_2^2
\right)
$$

Với $\mathbf{z}_i$ và $\mathbf{z}'_i$ lần lượt là biểu diễn trước và sau toán tử Koopman.

### 2.4. Quy trình huấn luyện

Mỗi epoch sử dụng toàn bộ 1.845 mẫu dữ liệu:

1. Thực hiện truyền xuôi để tạo dự báo.
2. Tính hàm mất mát tổng hợp.
3. Lan truyền ngược gradient.
4. Cập nhật trọng số bằng Adam.
5. Ghi lại loss và các giá trị kỳ dị của toán tử Koopman.

Giá trị loss được lưu ở mỗi epoch, trong khi các giá trị kỳ dị được ghi nhận định kỳ sau mỗi 20 epoch, bắt đầu từ epoch đầu tiên.

### 2.5. Kết quả đầu ra

Sau 3.000 epoch, checkpoint được lưu tại:

`results/patchtst_original_fullbatch.pt`

Checkpoint chứa:

- Trọng số của mô hình.
- Lịch sử loss.
- Lịch sử các giá trị kỳ dị.
- Các epoch ghi nhận giá trị kỳ dị.
- Số epoch, batch size và learning rate.

Phần đánh giá sau huấn luyện sử dụng chính bộ dữ liệu huấn luyện (*in-sample*), không phải tập kiểm tra độc lập.


In [ ]:
torch.manual_seed(42)

if device.type == "cuda":
    torch.cuda.manual_seed_all(42)
    torch.cuda.reset_peak_memory_stats()

# 1. Khởi tạo mô hình
patchtst_original_model = PatchTSTWrapper(
    patch_length=patch_length,
    horizon=horizon
).to(device)

# 2. Khởi tạo bộ tối ưu và hàm mất mát
opt = optim.Adam(patchtst_original_model.parameters(), lr=0.001)
mse = nn.MSELoss()

eig_hist = []
eig_epochs = []
loss_history = []

epochs = 3000

print("=== Original PatchTST Training ===")
print("Training samples:", len(x_in))
print("Batch size:", len(x_in))
print("Epochs:", epochs)
print("Optimizer: Adam")
print("Learning rate: 0.001")
print("Device:", device)

# 3. Huấn luyện full-batch
for e in range(epochs):

    patchtst_original_model.train()
    opt.zero_grad()

    # Sử dụng toàn bộ 1845 mẫu trong mỗi epoch
    preds, latent, latent_next, _, eigvals = patchtst_original_model(x_in)

    # Tính hàm mất mát gồm MSE và chính quy hóa ổn định
    loss = mse(preds, x_out) + 0.1 * torch.relu(
        latent_next.norm(dim=1) ** 2
        - latent.norm(dim=1) ** 2
    ).mean()

    loss.backward()
    opt.step()

    # Ghi lịch sử huấn luyện, không làm thay đổi quá trình tối ưu
    loss_history.append(loss.item())

    if e % 20 == 0:
        eig_hist.append(eigvals.detach().cpu().numpy().copy())
        eig_epochs.append(e + 1)

    if (e + 1) % 100 == 0 or e == 0:
        print(
            f"Epoch [{e+1}/{epochs}] "
            f"Loss: {loss.item():.6f}"
        )

# 4. Đánh giá trên chính dữ liệu huấn luyện
patchtst_original_model.eval()

with torch.no_grad():
    preds, _, _, _, _ = patchtst_original_model(x_in)

original_preds = preds.cpu().numpy()
original_targets = x_out.cpu().numpy()

patchtst_original_loss_history = np.array(loss_history)
patchtst_original_singular_history = np.array(eig_hist)
patchtst_original_singular_epochs = np.array(eig_epochs)

# 5. Lưu checkpoint và lịch sử huấn luyện
original_checkpoint_path = (
    experiment_dir / "results" / "patchtst_original_fullbatch.pt"
)

torch.save(
    {
        "model_state_dict": patchtst_original_model.state_dict(),
        "loss_history": patchtst_original_loss_history,
        "singular_history": patchtst_original_singular_history,
        "singular_epochs": patchtst_original_singular_epochs,
        "epochs": epochs,
        "batch_size": len(x_in),
        "learning_rate": 0.001
    },
    original_checkpoint_path
)

print("\n=== Final Results ===")
print("Initial Loss:", loss_history[0])
print("Final Loss:", loss_history[-1])
print("Final In-Sample MSE:", mse(preds, x_out).item())
print("Max Singular Value:", eig_hist[-1].max())
print("Checkpoint:", original_checkpoint_path)

if device.type == "cuda":
    print(
        "Peak GPU Memory (MiB):",
        round(torch.cuda.max_memory_allocated() / 1024**2, 2)
    )


=== Original PatchTST Training ===
Training samples: 1845
Batch size: 1845
Epochs: 3000
Optimizer: Adam
Learning rate: 0.001
Device: cuda
Epoch [1/3000] Loss: 278.294434
Epoch [100/3000] Loss: 250.262939
Epoch [200/3000] Loss: 208.407196
Epoch [300/3000] Loss: 161.409088
Epoch [400/3000] Loss: 129.810883
Epoch [500/3000] Loss: 73.278549
Epoch [600/3000] Loss: 42.512379
Epoch [700/3000] Loss: 25.108274
Epoch [800/3000] Loss: 15.989374
Epoch [900/3000] Loss: 11.051533
Epoch [1000/3000] Loss: 7.998667
Epoch [1100/3000] Loss: 5.855582
Epoch [1200/3000] Loss: 4.362207
Epoch [1300/3000] Loss: 3.363740
Epoch [1400/3000] Loss: 2.615838
Epoch [1500/3000] Loss: 2.111570
Epoch [1600/3000] Loss: 1.748827
Epoch [1700/3000] Loss: 1.497848
Epoch [1800/3000] Loss: 1.282714
Epoch [1900/3000] Loss: 1.134827
Epoch [2000/3000] Loss: 1.002944
Epoch [2100/3000] Loss: 0.900819
Epoch [2200/3000] Loss: 0.811149
Epoch [2300/3000] Loss: 0.740532
Epoch [2400/3000] Loss: 0.670925
Epoch [2500/3000] Loss: 0.601792
E


## 3. Huấn luyện mô hình Autoformer kết hợp Koopman

### 3.1. Mục tiêu

Thí nghiệm huấn luyện `AutoformerWrapper` để đánh giá cách một backbone có thành phần phân rã chuỗi thời gian hoạt động khi kết hợp với toán tử Koopman ổn định.

Mô hình sử dụng cùng bộ dữ liệu và cấu hình huấn luyện cơ bản với biến thể PatchTST.

### 3.2. Cấu hình huấn luyện

| Tham số | Giá trị |
|---|---|
| Mô hình | `AutoformerWrapper` |
| Số epoch | 3.000 |
| Batch size | 1.845 (full-batch) |
| Bộ tối ưu | Adam |
| Learning rate | 0.001 |
| Hệ số chính quy hóa | 0.1 |
| Random seed | 42 |

### 3.3. Quy trình

Mỗi epoch thực hiện truyền xuôi, tính hàm mất mát kết hợp MSE và chính quy hóa ổn định, lan truyền ngược và cập nhật trọng số.

Công thức hàm mất mát giống với phần huấn luyện PatchTST ở mục 2.3.

Trong quá trình tối ưu, loss được lưu tại mỗi epoch và các giá trị kỳ dị của toán tử Koopman được ghi nhận định kỳ.

### 3.4. Lưu kết quả

Checkpoint được lưu tại:

`results/autoformer_original_fullbatch.pt`

Tệp chứa trọng số mô hình và lịch sử huấn luyện.

Sau khi huấn luyện, notebook đánh giá mô hình trên chính bộ dữ liệu đã dùng để tối ưu.

**Lưu ý:** Các giá trị loss trong chế độ huấn luyện và MSE trong chế độ đánh giá có thể khác nhau do hành vi của các thành phần như Dropout. Phần này được phân tích riêng ở các cell tiếp theo.


In [3]:

torch.manual_seed(42)

if device.type == "cuda":
    torch.cuda.manual_seed_all(42)
    torch.cuda.reset_peak_memory_stats()

# 1. Initialize model
autoformer_original_model = AutoformerWrapper(
    patch_length=patch_length,
    horizon=horizon
).to(device)

# 2. Original optimizer and loss
opt = optim.Adam(autoformer_original_model.parameters(), lr=0.001)
mse = nn.MSELoss()

eig_hist = []
eig_epochs = []
loss_history = []

epochs = 3000

print("=== Original Autoformer Training ===")
print("Training samples:", len(x_in))
print("Batch size:", len(x_in))
print("Epochs:", epochs)
print("Optimizer: Adam")
print("Learning rate: 0.001")
print("Device:", device)

# 3. Original full-batch training loop
for e in range(epochs):

    autoformer_original_model.train()
    opt.zero_grad()

    # Full-batch: all 1845 samples
    preds, latent, latent_next, _, eigvals = autoformer_original_model(x_in)

    # Original loss
    loss = mse(preds, x_out) + 0.1 * torch.relu(
        latent_next.norm(dim=1) ** 2
        - latent.norm(dim=1) ** 2
    ).mean()

    loss.backward()
    opt.step()

    # Additional logging (does not alter training)
    loss_history.append(loss.item())

    if e % 20 == 0:
        eig_hist.append(eigvals.detach().cpu().numpy().copy())
        eig_epochs.append(e + 1)

    if (e + 1) % 100 == 0 or e == 0:
        print(
            f"Epoch [{e+1}/{epochs}] "
            f"Loss: {loss.item():.6f}"
        )

# 4. Original in-sample evaluation
autoformer_original_model.eval()

with torch.no_grad():
    preds, _, _, _, _ = autoformer_original_model(x_in)

autoformer_original_preds = preds.cpu().numpy()
autoformer_original_targets = x_out.cpu().numpy()

autoformer_original_loss_history = np.array(loss_history)
autoformer_original_singular_history = np.array(eig_hist)
autoformer_original_singular_epochs = np.array(eig_epochs)

# 5. Save checkpoint (additional step)
original_checkpoint_path = (
    experiment_dir / "results" / "autoformer_original_fullbatch.pt"
)

torch.save(
    {
        "model_state_dict": autoformer_original_model.state_dict(),
        "loss_history": autoformer_original_loss_history,
        "singular_history": autoformer_original_singular_history,
        "singular_epochs": autoformer_original_singular_epochs,
        "epochs": epochs,
        "batch_size": len(x_in),
        "learning_rate": 0.001
    },
    original_checkpoint_path
)

print("\n=== Final Results ===")
print("Initial Loss:", loss_history[0])
print("Final Loss:", loss_history[-1])
print("Final In-Sample MSE:", mse(preds, x_out).item())
print("Max Singular Value:", eig_hist[-1].max())
print("Checkpoint:", original_checkpoint_path)

if device.type == "cuda":
    print(
        "Peak GPU Memory (MiB):",
        round(torch.cuda.max_memory_allocated() / 1024**2, 2)
    )


=== Original Autoformer Training ===
Training samples: 1845
Batch size: 1845
Epochs: 3000
Optimizer: Adam
Learning rate: 0.001
Device: cuda:0
Epoch [1/3000] Loss: 277.696991
Epoch [100/3000] Loss: 243.754639
Epoch [200/3000] Loss: 199.721390
Epoch [300/3000] Loss: 158.843613
Epoch [400/3000] Loss: 469.767761
Epoch [500/3000] Loss: 106.090912
Epoch [600/3000] Loss: 88.119164
Epoch [700/3000] Loss: 75.571709
Epoch [800/3000] Loss: 72.827309
Epoch [900/3000] Loss: 72.321747
Epoch [1000/3000] Loss: 72.168404
Epoch [1100/3000] Loss: 72.171394
Epoch [1200/3000] Loss: 72.139740
Epoch [1300/3000] Loss: 72.125565
Epoch [1400/3000] Loss: 72.070702
Epoch [1500/3000] Loss: 71.671402
Epoch [1600/3000] Loss: 61.730644
Epoch [1700/3000] Loss: 42.512314
Epoch [1800/3000] Loss: 26.804863
Epoch [1900/3000] Loss: 19.774094
Epoch [2000/3000] Loss: 15.439191
Epoch [2100/3000] Loss: 12.570219
Epoch [2200/3000] Loss: 10.873465
Epoch [2300/3000] Loss: 9.321526
Epoch [2400/3000] Loss: 8.265129
Epoch [2500/3000


## 4. Huấn luyện mô hình Informer kết hợp Koopman

### 4.1. Mục tiêu

Thí nghiệm huấn luyện `InformerWrapper` trên cùng bộ dữ liệu Lorenz để đối chiếu với các biến thể sử dụng backbone dựa trên PatchTST và Autoformer.

Backbone trong mô hình này xử lý chuỗi đầu vào theo các bước thời gian, sau đó tạo biểu diễn tiềm ẩn và đưa qua toán tử Koopman ổn định.

### 4.2. Cấu hình huấn luyện

| Tham số | Giá trị |
|---|---|
| Mô hình | `InformerWrapper` |
| Số mẫu | 1.845 |
| Số epoch | 3.000 |
| Batch size | 1.845 (full-batch) |
| Bộ tối ưu | Adam |
| Learning rate | 0.001 |
| Hệ số chính quy hóa | 0.1 |
| Random seed | 42 |

### 4.3. Phương pháp

Mô hình được huấn luyện bằng phương pháp full-batch, với một lần cập nhật trọng số ở mỗi epoch.

Hàm mất mát kết hợp MSE và chính quy hóa ổn định, sử dụng cùng công thức đã trình bày tại mục 2.3.

Các giá trị loss và giá trị kỳ dị của toán tử Koopman được ghi nhận trong quá trình huấn luyện.

### 4.4. Lưu checkpoint

Sau huấn luyện, trọng số mô hình và lịch sử huấn luyện được lưu tại:

`results/informer_original_fullbatch.pt`

Mô hình được đánh giá trên chính bộ dữ liệu huấn luyện để tính MSE sau huấn luyện.

Kết quả này là **in-sample MSE**, không phải test MSE trên một tập dữ liệu độc lập.


In [3]:
torch.manual_seed(42)

if device.type == "cuda":
    torch.cuda.manual_seed_all(42)
    torch.cuda.reset_peak_memory_stats()

# 1. Initialize model
informer_original_model = InformerWrapper(
    patch_length=patch_length,
    horizon=horizon
).to(device)

# 2. Original optimizer and loss
opt = optim.Adam(informer_original_model.parameters(), lr=0.001)
mse = nn.MSELoss()

eig_hist = []
eig_epochs = []
loss_history = []

epochs = 3000

print("=== Original Informer Training ===")
print("Training samples:", len(x_in))
print("Batch size:", len(x_in))
print("Epochs:", epochs)
print("Optimizer: Adam")
print("Learning rate: 0.001")
print("Device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(device))

# 3. Original full-batch training loop
for e in range(epochs):

    informer_original_model.train()
    opt.zero_grad()

    # Full-batch: all 1845 samples
    preds, latent, latent_next, _, eigvals = informer_original_model(x_in)

    # Original loss
    loss = mse(preds, x_out) + 0.1 * torch.relu(
        latent_next.norm(dim=1) ** 2
        - latent.norm(dim=1) ** 2
    ).mean()

    loss.backward()
    opt.step()

    # Additional logging (does not alter training)
    loss_history.append(loss.item())

    if e % 20 == 0:
        eig_hist.append(eigvals.detach().cpu().numpy().copy())
        eig_epochs.append(e + 1)

    if (e + 1) % 100 == 0 or e == 0:
        print(
            f"Epoch [{e+1}/{epochs}] "
            f"Loss: {loss.item():.6f}"
        )

# 4. Save checkpoint
informer_original_loss_history = np.array(loss_history)
informer_original_singular_history = np.array(eig_hist)
informer_original_singular_epochs = np.array(eig_epochs)

original_checkpoint_path = (
    experiment_dir / "results" / "informer_original_fullbatch.pt"
)

torch.save(
    {
        "model_state_dict": informer_original_model.state_dict(),
        "loss_history": informer_original_loss_history,
        "singular_history": informer_original_singular_history,
        "singular_epochs": informer_original_singular_epochs,
        "epochs": epochs,
        "batch_size": len(x_in),
        "learning_rate": 0.001
    },
    original_checkpoint_path
)

# 5. Original in-sample evaluation
informer_original_model.eval()

with torch.no_grad():
    preds, _, _, _, _ = informer_original_model(x_in)

    _, _, final_singular_values = informer_original_model.koopman(
        torch.zeros(1, 16, device=device)
    )

informer_original_preds = preds.cpu().numpy()
informer_original_targets = x_out.cpu().numpy()

print("\n=== Final Results ===")
print("Initial Loss:", loss_history[0])
print("Final Loss:", loss_history[-1])
print("Final In-Sample MSE:", mse(preds, x_out).item())
print("Max Singular Value:", final_singular_values.max().item())
print("Checkpoint:", original_checkpoint_path)

if device.type == "cuda":
    print(
        "Peak GPU Memory (MiB):",
        round(torch.cuda.max_memory_allocated() / 1024**2, 2)
    )


=== Original Informer Training ===
Training samples: 1845
Batch size: 1845
Epochs: 3000
Optimizer: Adam
Learning rate: 0.001
Device: cuda
GPU: Quadro T1000
Epoch [1/3000] Loss: 282.345551
Epoch [100/3000] Loss: 244.040787
Epoch [200/3000] Loss: 203.136108
Epoch [300/3000] Loss: 157.021164
Epoch [400/3000] Loss: 106.965912
Epoch [500/3000] Loss: 60.992970
Epoch [600/3000] Loss: 41.711239
Epoch [700/3000] Loss: 30.417049
Epoch [800/3000] Loss: 28.009262
Epoch [900/3000] Loss: 27.304628
Epoch [1000/3000] Loss: 16.992786
Epoch [1100/3000] Loss: 10.876915
Epoch [1200/3000] Loss: 7.746890
Epoch [1300/3000] Loss: 6.331288
Epoch [1400/3000] Loss: 5.502367
Epoch [1500/3000] Loss: 4.838574
Epoch [1600/3000] Loss: 3.814125
Epoch [1700/3000] Loss: 2.589435
Epoch [1800/3000] Loss: 1.866870
Epoch [1900/3000] Loss: 1.529378
Epoch [2000/3000] Loss: 50.361977
Epoch [2100/3000] Loss: 37.136913
Epoch [2200/3000] Loss: 25.940798
Epoch [2300/3000] Loss: 11.282610
Epoch [2400/3000] Loss: 3.985792
Epoch [250